In [ ]:
import asyncio
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime, timedelta
import time
import os

# Add the project root to the path to import core modules
notebook_path = Path().absolute()
project_root = notebook_path.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from core.data_sources.clob import CLOBDataSource

async def fetch_historical_data(
    trading_pairs=['WLD-USDT', 'TRUMP-USDT', 'BTC-USDT', 'ETH-USDT', 'SOL-USDT', 'XRP-USDT', 'ADA-USDT','DOGE-USDT', 'AVAX-USDT', 'LINK-USDT', 'DOT-USDT', 'MATIC-USDT', 'USDC-USDT','LTC-USDT', 'UNI-USDT', 'BCH-USDT', 'ATOM-USDT', 'ETC-USDT', 'FIL-USDT','NEAR-USDT', 'APE-USDT', 'AAVE-USDT', 'AXS-USDT', 'ALGO-USDT', 'APT-USDT','ARB-USDT', 'COMP-USDT', 'DASH-USDT', 'EOS-USDT', 'FTM-USDT', 'GALA-USDT','ICP-USDT', 'MANA-USDT', 'MASK-USDT', 'RNDR-USDT', 'SAND-USDT', 'SHIB-USDT','SUI-USDT', 'TRX-USDT', 'VET-USDT', 'XLM-USDT', 'XMR-USDT', 'ZEC-USDT','OP-USDT', 'PEPE-USDT', 'INJ-USDT', 'BLUR-USDT', 'DYDX-USDT', 'GMX-USDT','IMX-USDT', 'KDA-USDT', 'PROM-USDT', 'AVA-USDT', 'BNB-USDT'],
    intervals=['1s'],
    days=365
):
    """Fetch historical candlestick data using existing CLOB infrastructure"""
    clob = CLOBDataSource()
    output_dir = Path('candlestick_data')
    output_dir.mkdir(exist_ok=True)

    connector_name = "binance"
    for trading_pair in trading_pairs:
        for interval in intervals:
            try:
                print(f"Fetching {interval} data for {trading_pair}...")
                candles = await clob.get_candles_last_days(
                    connector_name=connector_name,
                    trading_pair=trading_pair,
                    interval=interval,
                    days=days
                )
                # Save to Parquet with requested format
                filename = output_dir / f"{connector_name}|{trading_pair}|{interval}.parquet"
                candles.data.to_parquet(filename)
                print(f"Saved {filename}")
            except Exception as e:
                print(f"Error fetching {interval} data for {trading_pair}: {str(e)}")

    clob.dump_candles_cache()

# For Jupyter notebook execution
async def run_fetch():
    await fetch_historical_data()
    print("Data collection complete!")

# Execute in notebook
await run_fetch()